In [1]:
import os
import numpy as np
import pandas as pd

# Relative to Our Notebooks/
PROVIDED_DIR = "../Provided Datasets"
NEW_DIR = "../New Datasets"
OUTPUT_DIR = NEW_DIR + "/Combined"
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Expected training files
EXPECTED_FILES = {
    "gaia": "gaia_features_training.csv",
    "jrc_gsw": "jrc_gsw_features_training.csv",
    "landsat_allbands": "landsat_features_training_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_training_allvars.csv",
    "esa_cci": "esa_cci_features_training.csv",
    "raster_buffer": "esa_jrc_gaia_buffer_training.csv",
}

def resolve_path(fname: str) -> str | None:
    """Return the first existing path for fname across Provided and New dirs."""
    for base in (PROVIDED_DIR, NEW_DIR):
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return None

resolved = {}
missing = []
for key, fname in EXPECTED_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing.append((key, fname))
    else:
        resolved[key] = p

print("Resolved input paths:")
for k, p in resolved.items():
    print(f" - {k:18s} -> {os.path.abspath(p)}")

if missing:
    msg = "Missing these expected files in BOTH folders:\n" + "\n".join([f"{k}: {f}" for k, f in missing])
    raise FileNotFoundError(msg)

print("\nAll expected files found")

In [3]:
def standardize_join_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize join columns to: latitude, longitude, sample_date."""
    out = df.copy()
    rename_map = {}
    for c in out.columns:
        lc = c.strip().lower()
        if lc in {"latitude", "lat"} or lc.startswith("lat"):
            rename_map[c] = "latitude"
        elif lc in {"longitude", "lon", "lng"} or lc.startswith("lon"):
            rename_map[c] = "longitude"
        # Be conservative: only map obvious sample-date columns
        elif lc in {"sample date", "sample_date"}:
            rename_map[c] = "sample_date"
    out = out.rename(columns=rename_map)

    # If a dataset used a generic "Date" column, map it ONLY if sample_date doesn't exist yet
    if "sample_date" not in out.columns:
        for c in list(out.columns):
            if c.strip().lower() == "date":
                out = out.rename(columns={c: "sample_date"})
                break

    # Drop duplicate columns created by renaming collisions
    out = out.loc[:, ~out.columns.duplicated(keep="first")]

    required = {"latitude", "longitude", "sample_date"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(f"Missing required join columns after standardization: {missing}")

    # helper parsed date (not used for join)
    out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
    return out

In [ ]:
# Load + standardize
datasets = {}
for name, path in resolved.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    datasets[name] = df_std
    print(f"{name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")

In [5]:
# Proper merge on join keys + output Final Datasets
merge_keys = ["latitude", "longitude", "sample_date"]

VALIDATION_FILES = {
    "gaia": "gaia_features_validation.csv",
    "jrc_gsw": "jrc_gsw_features_validation.csv",
    "landsat_allbands": "landsat_features_validation_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_validation_allvars.csv",
    "esa_cci": "esa_cci_features_validation.csv",
    "raster_buffer": "esa_jrc_gaia_buffer_validation.csv",
}

resolved_val = {}
missing_val = []
for key, fname in VALIDATION_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing_val.append((key, fname))
    else:
        resolved_val[key] = p

if missing_val:
    print("Missing validation files:")
    for k, f in missing_val:
        print(f" - {k}: {f}")
    raise FileNotFoundError("Missing one or more validation feature files")

# Load + standardize validation datasets
val_datasets = {}
for name, path in resolved_val.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    val_datasets[name] = df_std
    print(f"VAL {name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")


def merge_feature_datasets(datasets_dict):
    merged_out = None
    for name, df in datasets_dict.items():
        if merged_out is None:
            merged_out = df
        else:
            merged_out = pd.merge(
                merged_out,
                df,
                on=merge_keys,
                how="outer",
                suffixes=("", f"__{name}")
            )

    cols_to_drop = [
        c for c in merged_out.columns
        if ("date" in c.lower()) and (c.lower() != "sample_date")
    ]
    if cols_to_drop:
        print("Dropping from dataframe:")
        print(cols_to_drop)
        merged_out = merged_out.drop(columns=cols_to_drop)
    return merged_out

# Merge training + validation feature datasets
merged_train = merge_feature_datasets(datasets)
merged_val = merge_feature_datasets(val_datasets)

print("Final training feature shape:", merged_train.shape)
print("Final validation feature shape:", merged_val.shape)

# --- Proper join with targets (3-key) ---
wq_data = pd.read_csv(os.path.join(PROJECT_ROOT, "Provided Datasets", "water_quality_training_dataset.csv"))
wq_data.columns = wq_data.columns.str.lower().str.replace(" ", "_")

merged_train["sample_date"] = pd.to_datetime(merged_train["sample_date"], format="%d-%m-%Y", errors="coerce")
wq_data["sample_date"] = pd.to_datetime(wq_data["sample_date"], format="%d-%m-%Y", errors="coerce")

merged_train["latitude"] = merged_train["latitude"].round(6)
merged_train["longitude"] = merged_train["longitude"].round(6)
wq_data["latitude"] = wq_data["latitude"].round(6)
wq_data["longitude"] = wq_data["longitude"].round(6)

# Handle censored DRP
DRP_LOD_10 = 10.0
DRP_LOD_20 = 20.0

def prepare_drp_for_ml(df, drp_col="dissolved_reactive_phosphorus", random_state=42):
    out = df.copy()
    rng = np.random.default_rng(random_state)
    drp = out[drp_col].astype(float)
    out["drp_censored_10"] = (drp == DRP_LOD_10).astype(int)
    out["drp_censored_20"] = (drp == DRP_LOD_20).astype(int)
    mask_10 = (drp == DRP_LOD_10)
    mask_20 = (drp == DRP_LOD_20)
    out.loc[mask_10, drp_col] = rng.uniform(0, DRP_LOD_10, size=mask_10.sum())
    out.loc[mask_20, drp_col] = rng.uniform(DRP_LOD_10, DRP_LOD_20, size=mask_20.sum())
    return out

wq_data = prepare_drp_for_ml(wq_data)

wq_targets = wq_data[[
    "latitude", "longitude", "sample_date",
    "total_alkalinity", "electrical_conductance", "dissolved_reactive_phosphorus",
    "drp_censored_10", "drp_censored_20",
]].copy()

complete_data = merged_train.merge(
    wq_targets,
    on=["latitude", "longitude", "sample_date"],
    how="inner"
)

# Feature list (exclude join keys)
feature_cols = [c for c in merged_train.columns if c not in ["latitude", "longitude", "sample_date"]]

# month_fitted
from pathlib import Path
import pickle
params_path = Path("Our Notebooks/month_engineering/fitted_monthly_params.pkl")
with open(params_path, "rb") as f:
    fitted_params = pickle.load(f)

complete_data["month"] = pd.to_datetime(complete_data["sample_date"]).dt.month
merged_val["month"] = pd.to_datetime(merged_val["sample_date"], format="%d-%m-%Y", errors="coerce").dt.month

# DRP
params = fitted_params["DRP"]["params"]
complete_data["month_fitted_drp"] = (
    params[0] * complete_data["month"]**3 +
    params[1] * complete_data["month"]**2 +
    params[2] * complete_data["month"] +
    params[3]
)
merged_val["month_fitted_drp"] = (
    params[0] * merged_val["month"]**3 +
    params[1] * merged_val["month"]**2 +
    params[2] * merged_val["month"] +
    params[3]
)

# EC
params = fitted_params["EC"]["params"]
complete_data["month_fitted_ec"] = (
    params[0] * np.sin(params[1] * complete_data["month"] + params[2]) + params[3]
)
merged_val["month_fitted_ec"] = (
    params[0] * np.sin(params[1] * merged_val["month"] + params[2]) + params[3]
)

# TA
params = fitted_params["TA"]["params"]
complete_data["month_fitted_ta"] = (
    params[0] * np.sin(params[1] * complete_data["month"] + params[2]) + params[3]
)
merged_val["month_fitted_ta"] = (
    params[0] * np.sin(params[1] * merged_val["month"] + params[2]) + params[3]
)

# Create datasets
output_path = Path("New Datasets/Combined/Final Datasets")
output_path.mkdir(parents=True, exist_ok=True)

# Training
train_base = complete_data[["latitude", "longitude"] + feature_cols].copy()

# DRP
train_drp = train_base.copy()
train_drp["dissolved_reactive_phosphorus"] = complete_data["dissolved_reactive_phosphorus"]
train_drp["drp_censored_10"] = complete_data["drp_censored_10"]
train_drp["drp_censored_20"] = complete_data["drp_censored_20"]
train_drp["month_fitted"] = complete_data["month_fitted_drp"]

# EC
train_ec = train_base.copy()
train_ec["electrical_conductance"] = complete_data["electrical_conductance"]
train_ec["month_fitted"] = complete_data["month_fitted_ec"]

# TA
train_ta = train_base.copy()
train_ta["total_alkalinity"] = complete_data["total_alkalinity"]
train_ta["month_fitted"] = complete_data["month_fitted_ta"]

# Validation
val_base = merged_val[["latitude", "longitude"] + feature_cols].copy()

val_drp = val_base.copy()
val_drp["month_fitted"] = merged_val["month_fitted_drp"]

val_ec = val_base.copy()
val_ec["month_fitted"] = merged_val["month_fitted_ec"]

val_ta = val_base.copy()
val_ta["month_fitted"] = merged_val["month_fitted_ta"]

# Save
train_drp.to_csv(output_path / "drp_training_complete.csv", index=False)
train_ec.to_csv(output_path / "ec_training_complete.csv", index=False)
train_ta.to_csv(output_path / "ta_training_complete.csv", index=False)

val_drp.to_csv(output_path / "drp_validation.csv", index=False)
val_ec.to_csv(output_path / "ec_validation.csv", index=False)
val_ta.to_csv(output_path / "ta_validation.csv", index=False)


Dropping from dataframe:
['sample_date_parsed', 'sample_date_parsed__jrc_gsw', 'sample_date_parsed__landsat_allbands', 'sample_date_parsed__terraclimate_allvars', 'sample_date_parsed__esa_cci', 'sample_date_parsed__raster_buffer']
Final shape: (9319, 70)


In [6]:
# shape checking (Final Datasets)
from pathlib import Path

final_dir = Path("New Datasets/Combined/Final Datasets")

train_drp = pd.read_csv(final_dir / "drp_training_complete.csv")
train_ec = pd.read_csv(final_dir / "ec_training_complete.csv")
train_ta = pd.read_csv(final_dir / "ta_training_complete.csv")

val_drp = pd.read_csv(final_dir / "drp_validation.csv")
val_ec = pd.read_csv(final_dir / "ec_validation.csv")
val_ta = pd.read_csv(final_dir / "ta_validation.csv")

print("Training shapes:")
print("  DRP:", train_drp.shape)
print("  EC :", train_ec.shape)
print("  TA :", train_ta.shape)

print("\nValidation shapes:")
print("  DRP:", val_drp.shape)
print("  EC :", val_ec.shape)
print("  TA :", val_ta.shape)

print("\nFeature column counts (train):")
print("  DRP:", train_drp.drop(columns=["dissolved_reactive_phosphorus", "drp_censored_10", "drp_censored_20"]).shape[1])
print("  EC :", train_ec.drop(columns=["electrical_conductance"]).shape[1])
print("  TA :", train_ta.drop(columns=["total_alkalinity"]).shape[1])

print("\nFeature column counts (val):")
print("  DRP:", val_drp.shape[1])
print("  EC :", val_ec.shape[1])
print("  TA :", val_ta.shape[1])

    mismatch_rows = []
    for col in df_std.columns:
        if col in merge_keys or col == "_row_id":
            continue
        merged_col = _col_in_merged(col, name)
        if merged_col is None:
            mismatch_rows.append({"dataset": name, "column": col, "issue": "missing_in_merged"})
            continue

        src_col = f"{col}__src"
        if src_col not in merged_with.columns:
            mismatch_rows.append({"dataset": name, "column": col, "issue": "missing_in_source_after_merge"})
            continue

        equal_mask = _equal_series(merged_with[merged_col], merged_with[src_col], tol=tol)
        if not bool(equal_mask.all()):
            mismatch_rows.append({
                "dataset": name,
                "column": col,
                "issue": "value_mismatch",
                "mismatch_count": int((~equal_mask).sum()),
            })

    return pd.DataFrame(mismatch_rows)


mismatch_reports = []
for name, df_std in datasets.items():
    report = validate_dataset(name, df_std)
    mismatch_reports.append(report)

mismatch_summary = pd.concat(mismatch_reports, ignore_index=True) if mismatch_reports else pd.DataFrame()
print("Mismatch summary (empty means all checks passed):")
display(mismatch_summary)

Real Shape: 48
Merged Shape: 70
Mismatch summary (empty means all checks passed):


""


In [7]:
# dropping features
dropped_cols = [
    "qa_pixel",
    "qa_aerosol",
    "esa_processed_flag",
    "esa_observation_count",
    "esa_current_pixel_state",
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus",
    "coastal",
    "lwir11"
]

cols_to_drop = [c for c in dropped_cols if c in merged.columns]
filtered_no_pixel_flags = merged.drop(columns=cols_to_drop).copy()

# Remove raster buffer features from the "normal" dataset
raster_cols = [c for c in filtered_no_pixel_flags.columns if c.endswith("_1km")]
filtered_no_pixel_flags = filtered_no_pixel_flags.drop(columns=raster_cols)

print("Dropped columns:", cols_to_drop)
print("Dropped raster columns:", raster_cols)
print(filtered_no_pixel_flags.shape)

# Save normal (non-engineered) dataset
out_no_flags = os.path.join(OUTPUT_DIR, "combined_training_dataset.csv")
filtered_no_pixel_flags.to_csv(out_no_flags, index=False)
print("Saved:", os.path.abspath(out_no_flags))

Dropped columns: ['qa_pixel', 'qa_aerosol', 'esa_processed_flag', 'esa_observation_count', 'esa_current_pixel_state', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'coastal', 'lwir11']
Dropped raster columns: ['esa_urban_frac_1km', 'esa_cropland_frac_1km', 'esa_water_frac_1km', 'esa_forest_frac_1km', 'esa_shrub_frac_1km', 'esa_grass_frac_1km', 'esa_sparse_veg_frac_1km', 'esa_bare_frac_1km', 'esa_snow_ice_frac_1km', 'esa_flooded_frac_1km', 'esa_other_frac_1km', 'gsw_occurrence_mean_1km', 'gsw_seasonality_mean_1km', 'gsw_recurrence_mean_1km', 'gsw_extent_mean_1km', 'gsw_change_mean_1km', 'gsw_water_frac_1km', 'gaia_changed_ever_frac_1km', 'gaia_impervious_frac_by_sample_year_1km', 'gaia_recent_change_5y_frac_1km', 'gaia_years_since_change_mean_1km', 'gaia_transition_year_mean_changed_pixels_1km']
(9319, 38)
Saved: /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/Combined/combined_training_dataset.csv


In [8]:
# adding feature to handle landsat nulls
# Landsat columns to check for missingness
landsat_targets = ["nir","ndmi","blue","red","swir22","swir16","green","mndwi"]

merged_cols_lower = {c.lower(): c for c in filtered_no_pixel_flags.columns}
resolved_cols = []
for t in landsat_targets:
    if t in merged_cols_lower:
        resolved_cols.append(merged_cols_lower[t])
        continue
    alt = f"{t}__landsat"
    if alt in merged_cols_lower:
        resolved_cols.append(merged_cols_lower[alt])
        continue

missing = [t for t, c in zip(landsat_targets, resolved_cols + [None] * (len(landsat_targets) - len(resolved_cols))) if c is None]
if missing:
    raise ValueError(f"Missing expected Landsat columns in filtered_no_pixel_flags: {missing}")

# We compute landsat_present later in the engineering cell
print("Landsat columns found:", resolved_cols)
print("(landsat_present will be created in the engineering step)")

Landsat columns found: ['nir', 'NDMI', 'blue', 'red', 'swir22', 'swir16', 'green', 'MNDWI']
(landsat_present will be created in the engineering step)


In [9]:
# data engineering
# Start from normal dataset and add raster buffer columns back in
merge_keys = ["latitude", "longitude", "sample_date"]
raster_cols = [c for c in merged.columns if c.endswith("_1km")]

engineered = filtered_no_pixel_flags.merge(
    merged[merge_keys + raster_cols],
    on=merge_keys,
    how="left",
)

EPS = 1e-6

# --- Date cyclic features ---
if "sample_date" in engineered.columns:
    dates = pd.to_datetime(engineered["sample_date"], dayfirst=True, errors="coerce")
    months = dates.dt.month.fillna(1).astype(int)
    engineered["month_sin"] = np.sin(2 * np.pi * months / 12)
    engineered["month_cos"] = np.cos(2 * np.pi * months / 12)

# --- Landsat features ---
if "landsat_present" not in engineered.columns:
    landsat_cols = [c for c in ["nir","ndmi","blue","red","swir22","swir16","green","mndwi"] if c in engineered.columns]
    if landsat_cols:
        engineered["landsat_present"] = (~engineered[landsat_cols].isna().any(axis=1)).astype(int)

if all(c in engineered.columns for c in ["nir", "red"]):
    engineered["ndvi"] = (engineered["nir"] - engineered["red"]) / (engineered["nir"] + engineered["red"] + EPS)

if all(c in engineered.columns for c in ["nir", "swir22"]):
    engineered["nbr"] = (engineered["nir"] - engineered["swir22"]) / (engineered["nir"] + engineered["swir22"] + EPS)

if all(c in engineered.columns for c in ["swir16", "swir22"]):
    engineered["mineral_index"] = (engineered["swir16"] - engineered["swir22"]) / (engineered["swir16"] + engineered["swir22"] + EPS)
    engineered["swir_ratio"] = engineered["swir16"] / (engineered["swir22"] + EPS)

if all(c in engineered.columns for c in ["swir22", "nir"]):
    engineered["salinity_proxy"] = engineered["swir22"] / (engineered["nir"] + EPS)

# --- TerraClimate features ---
if all(c in engineered.columns for c in ["ppt", "pet"]):
    engineered["wb"] = engineered["ppt"] - engineered["pet"]
    engineered["eci"] = engineered["pet"] / (engineered["ppt"] + EPS)

if all(c in engineered.columns for c in ["q", "ppt"]):
    engineered["rr"] = engineered["q"] / (engineered["ppt"] + EPS)

if all(c in engineered.columns for c in ["pet", "aet"]):
    engineered["etgap"] = engineered["pet"] - engineered["aet"]

if all(c in engineered.columns for c in ["def", "ppt"]):
    engineered["dsi"] = engineered["def"] / (engineered["ppt"] + EPS)

if all(c in engineered.columns for c in ["srad", "tmax", "tmin"]):
    engineered["hri"] = engineered["srad"] * ((engineered["tmax"] + engineered["tmin"]) / 2)

# --- JRC buffer features (if present) ---
if all(c in engineered.columns for c in ["gsw_occurrence_mean_1km", "gsw_seasonality_mean_1km"]):
    engineered["water_perm"] = engineered["gsw_occurrence_mean_1km"] * (engineered["gsw_seasonality_mean_1km"] / 12)
    engineered["water_instab"] = (1 - engineered["gsw_occurrence_mean_1km"] / 100) * (engineered["gsw_seasonality_mean_1km"] / 12)

if all(c in engineered.columns for c in ["gsw_recurrence_mean_1km", "gsw_extent_mean_1km"]):
    engineered["recurrence_ratio"] = engineered["gsw_recurrence_mean_1km"] / (engineered["gsw_extent_mean_1km"] + EPS)

if "gsw_seasonality_mean_1km" in engineered.columns:
    engineered["seasonal_water"] = engineered["gsw_seasonality_mean_1km"].between(1, 9).astype(int)

# --- ESA point-based features (if present) ---
if "esa_change_count" in engineered.columns:
    engineered["esa_change_intensity"] = np.log1p(engineered["esa_change_count"])

# --- GAIA buffer interactions (if present) ---
if all(c in engineered.columns for c in ["wb", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered["wb_x_impervious"] = engineered["wb"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

if all(c in engineered.columns for c in ["eci", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered["eci_x_impervious"] = engineered["eci"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

if all(c in engineered.columns for c in ["gsw_occurrence_mean_1km", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered["gsw_occ_x_impervious"] = engineered["gsw_occurrence_mean_1km"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

# Save
engineered_out = os.path.join(OUTPUT_DIR, "combined_training_engineered.csv")
engineered.to_csv(engineered_out, index=False)

print("Saved:", os.path.abspath(engineered_out))
print("Engineered shape:", engineered.shape)

Saved: /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/Combined/combined_training_engineered.csv
Engineered shape: (9319, 82)


In [10]:
# === Validation: resolve input feature files ===

EXPECTED_VAL_FILES = {
    "gaia": "gaia_features_validation.csv",
    "jrc_gsw": "jrc_gsw_features_validation.csv",
    "landsat_allbands": "landsat_features_validation_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_validation_allvars.csv",
    "esa_cci": "esa_cci_features_validation.csv",
    "raster_buffer": "esa_jrc_gaia_buffer_validation.csv",
}

resolved_val = {}
missing_val = []
for key, fname in EXPECTED_VAL_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing_val.append((key, fname))
    else:
        resolved_val[key] = p

print("Resolved VALIDATION input paths:")
for k, p in resolved_val.items():
    print(f" - {k:18s} -> {os.path.abspath(p)}")

if missing_val:
    msg = "Missing these expected VALIDATION files in BOTH folders:\n" + "\n".join([f"{k}: {f}" for k, f in missing_val])
    raise FileNotFoundError(msg)

print("\nAll expected VALIDATION files found")

Resolved VALIDATION input paths:
 - gaia               -> /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/gaia_features_validation.csv
 - jrc_gsw            -> /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/jrc_gsw_features_validation.csv
 - landsat_allbands   -> /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/landsat_features_validation_allbands.csv
 - terraclimate_allvars -> /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/terraclimate_features_validation_allvars.csv
 - esa_cci            -> /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/esa_cci_features_validation.csv
 - raster_buffer      -> /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/esa_jrc_gaia_buffer_validation.csv

All expected VALIDATION files found


In [11]:
# === Validation: load and standardize ===

datasets_val = {}
for name, path in resolved_val.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    datasets_val[name] = df_std
    print(f"{name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")

gaia               shape=(200, 9)  cols=9
jrc_gsw            shape=(200, 13)  cols=13
landsat_allbands   shape=(200, 16)  cols=16
terraclimate_allvars shape=(200, 18)  cols=18
esa_cci            shape=(200, 9)  cols=9
raster_buffer      shape=(200, 26)  cols=26


/var/folders/mc/q08d_s811l54wr44n0f4yhnw0000gp/T/ipykernel_92035/2031521441.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
/var/folders/mc/q08d_s811l54wr44n0f4yhnw0000gp/T/ipykernel_92035/2031521441.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
/var/folders/mc/q08d_s811l54wr44n0f4yhnw0000gp/T/ipykernel_92035/2031521441.py:32: UserW

In [12]:
# === Validation: outer merge on join keys ===

merge_keys = ["latitude", "longitude", "sample_date"]

merged_val = None
for name, df in datasets_val.items():
    if merged_val is None:
        merged_val = df
    else:
        merged_val = pd.merge(
            merged_val,
            df,
            on=merge_keys,
            how="outer",
            suffixes=("", f"__{name}")
        )

# Drop extra date-like columns as in training
merged_val = remove_extra_date_columns(merged_val)

print("Final VALIDATION merged shape:", merged_val.shape)

Dropping from dataframe:
['sample_date_parsed', 'sample_date_parsed__jrc_gsw', 'sample_date_parsed__landsat_allbands', 'sample_date_parsed__terraclimate_allvars', 'sample_date_parsed__esa_cci', 'sample_date_parsed__raster_buffer']
Final VALIDATION merged shape: (200, 70)


In [13]:
# === Validation: drop same feature sets and save non-engineered combined dataset ===

val_dropped_cols = [
    "qa_pixel",
    "qa_aerosol",
    "esa_processed_flag",
    "esa_observation_count",
    "esa_current_pixel_state",
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus",
    "coastal",
    "lwir11",
]

cols_to_drop_val = [c for c in val_dropped_cols if c in merged_val.columns]
filtered_no_pixel_flags_val = merged_val.drop(columns=cols_to_drop_val).copy()

# Remove raster buffer features from the "normal" validation dataset
raster_cols_val = [c for c in filtered_no_pixel_flags_val.columns if c.endswith("_1km")]
filtered_no_pixel_flags_val = filtered_no_pixel_flags_val.drop(columns=raster_cols_val)

print("VALIDATION - Dropped columns:", cols_to_drop_val)
print("VALIDATION - Dropped raster columns:", raster_cols_val)
print("VALIDATION filtered_no_pixel_flags shape:", filtered_no_pixel_flags_val.shape)

out_no_flags_val = os.path.join(OUTPUT_DIR, "combined_validation_dataset.csv")
filtered_no_pixel_flags_val.to_csv(out_no_flags_val, index=False)
print("Saved:", os.path.abspath(out_no_flags_val))

VALIDATION - Dropped columns: ['qa_pixel', 'qa_aerosol', 'esa_processed_flag', 'esa_observation_count', 'esa_current_pixel_state', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'coastal', 'lwir11']
VALIDATION - Dropped raster columns: ['esa_urban_frac_1km', 'esa_cropland_frac_1km', 'esa_water_frac_1km', 'esa_forest_frac_1km', 'esa_shrub_frac_1km', 'esa_grass_frac_1km', 'esa_sparse_veg_frac_1km', 'esa_bare_frac_1km', 'esa_snow_ice_frac_1km', 'esa_flooded_frac_1km', 'esa_other_frac_1km', 'gsw_occurrence_mean_1km', 'gsw_seasonality_mean_1km', 'gsw_recurrence_mean_1km', 'gsw_extent_mean_1km', 'gsw_change_mean_1km', 'gsw_water_frac_1km', 'gaia_changed_ever_frac_1km', 'gaia_impervious_frac_by_sample_year_1km', 'gaia_recent_change_5y_frac_1km', 'gaia_years_since_change_mean_1km', 'gaia_transition_year_mean_changed_pixels_1km']
VALIDATION filtered_no_pixel_flags shape: (200, 38)
Saved: /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/Co

In [14]:
# === Validation: engineering and save engineered combined dataset ===

merge_keys = ["latitude", "longitude", "sample_date"]
raster_cols_val = [c for c in merged_val.columns if c.endswith("_1km")]

engineered_val = filtered_no_pixel_flags_val.merge(
    merged_val[merge_keys + raster_cols_val],
    on=merge_keys,
    how="left",
)

if "EPS" not in globals():
    EPS = 1e-6

# --- Date cyclic features ---
if "sample_date" in engineered_val.columns:
    dates = pd.to_datetime(engineered_val["sample_date"], dayfirst=True, errors="coerce")
    months = dates.dt.month.fillna(1).astype(int)
    engineered_val["month_sin"] = np.sin(2 * np.pi * months / 12)
    engineered_val["month_cos"] = np.cos(2 * np.pi * months / 12)

# --- Landsat features ---
if "landsat_present" not in engineered_val.columns:
    landsat_cols = [c for c in ["nir","ndmi","blue","red","swir22","swir16","green","mndwi"] if c in engineered_val.columns]
    if landsat_cols:
        engineered_val["landsat_present"] = (~engineered_val[landsat_cols].isna().any(axis=1)).astype(int)

if all(c in engineered_val.columns for c in ["nir", "red"]):
    engineered_val["ndvi"] = (engineered_val["nir"] - engineered_val["red"]) / (engineered_val["nir"] + engineered_val["red"] + EPS)

if all(c in engineered_val.columns for c in ["nir", "swir22"]):
    engineered_val["nbr"] = (engineered_val["nir"] - engineered_val["swir22"]) / (engineered_val["nir"] + engineered_val["swir22"] + EPS)

if all(c in engineered_val.columns for c in ["swir16", "swir22"]):
    engineered_val["mineral_index"] = (engineered_val["swir16"] - engineered_val["swir22"]) / (engineered_val["swir16"] + engineered_val["swir22"] + EPS)
    engineered_val["swir_ratio"] = engineered_val["swir16"] / (engineered_val["swir22"] + EPS)

if all(c in engineered_val.columns for c in ["swir22", "nir"]):
    engineered_val["salinity_proxy"] = engineered_val["swir22"] / (engineered_val["nir"] + EPS)

# --- TerraClimate features ---
if all(c in engineered_val.columns for c in ["ppt", "pet"]):
    engineered_val["wb"] = engineered_val["ppt"] - engineered_val["pet"]
    engineered_val["eci"] = engineered_val["pet"] / (engineered_val["ppt"] + EPS)

if all(c in engineered_val.columns for c in ["q", "ppt"]):
    engineered_val["rr"] = engineered_val["q"] / (engineered_val["ppt"] + EPS)

if all(c in engineered_val.columns for c in ["pet", "aet"]):
    engineered_val["etgap"] = engineered_val["pet"] - engineered_val["aet"]

if all(c in engineered_val.columns for c in ["def", "ppt"]):
    engineered_val["dsi"] = engineered_val["def"] / (engineered_val["ppt"] + EPS)

if all(c in engineered_val.columns for c in ["srad", "tmax", "tmin"]):
    engineered_val["hri"] = engineered_val["srad"] * ((engineered_val["tmax"] + engineered_val["tmin"]) / 2)

# --- JRC buffer features (if present) ---
if all(c in engineered_val.columns for c in ["gsw_occurrence_mean_1km", "gsw_seasonality_mean_1km"]):
    engineered_val["water_perm"] = engineered_val["gsw_occurrence_mean_1km"] * (engineered_val["gsw_seasonality_mean_1km"] / 12)
    engineered_val["water_instab"] = (1 - engineered_val["gsw_occurrence_mean_1km"] / 100) * (engineered_val["gsw_seasonality_mean_1km"] / 12)

if all(c in engineered_val.columns for c in ["gsw_recurrence_mean_1km", "gsw_extent_mean_1km"]):
    engineered_val["recurrence_ratio"] = engineered_val["gsw_recurrence_mean_1km"] / (engineered_val["gsw_extent_mean_1km"] + EPS)

if "gsw_seasonality_mean_1km" in engineered_val.columns:
    engineered_val["seasonal_water"] = engineered_val["gsw_seasonality_mean_1km"].between(1, 9).astype(int)

# --- ESA point-based features (if present) ---
if "esa_change_count" in engineered_val.columns:
    engineered_val["esa_change_intensity"] = np.log1p(engineered_val["esa_change_count"])

# --- GAIA buffer interactions (if present) ---
if all(c in engineered_val.columns for c in ["wb", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered_val["wb_x_impervious"] = engineered_val["wb"] * engineered_val["gaia_impervious_frac_by_sample_year_1km"]

if all(c in engineered_val.columns for c in ["eci", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered_val["eci_x_impervious"] = engineered_val["eci"] * engineered_val["gaia_impervious_frac_by_sample_year_1km"]

if all(c in engineered_val.columns for c in ["gsw_occurrence_mean_1km", "gaia_impervious_frac_by_sample_year_1km"]):
    engineered_val["gsw_occ_x_impervious"] = engineered_val["gsw_occurrence_mean_1km"] * engineered_val["gaia_impervious_frac_by_sample_year_1km"]

# Save
engineered_val_out = os.path.join(OUTPUT_DIR, "combined_validation_engineered.csv")
engineered_val.to_csv(engineered_val_out, index=False)

print("Saved:", os.path.abspath(engineered_val_out))
print("Engineered VALIDATION shape:", engineered_val.shape)

Saved: /Users/ethan/Documents/EY Challenge/Water-Quality-Prediction/New Datasets/Combined/combined_validation_engineered.csv
Engineered VALIDATION shape: (200, 82)


In [15]:
# Correlation matrix + highly correlated feature pairs

numeric_cols = engineered.select_dtypes(include=[np.number]).columns
corr = engineered[numeric_cols].corr()

# Full correlation matrix (can be large)
print("Correlation matrix shape:", corr.shape)
display(corr)

# List top absolute correlations (excluding self-correlation)
abs_corr = corr.abs()
np.fill_diagonal(abs_corr.values, 0)

pairs = (
    abs_corr.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_corr"})
)

# Drop duplicate pairs (A,B) vs (B,A)
pairs = pairs[pairs["feature_a"] < pairs["feature_b"]]

# Show pairs above threshold
threshold = 0.9
high_corr = pairs[pairs["abs_corr"] >= threshold].sort_values("abs_corr", ascending=False)
print(f"Highly correlated pairs (|r| >= {threshold}): {len(high_corr)}")
display(high_corr)


Correlation matrix shape: (81, 81)


,latitude,longitude,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,gsw_occurrence,...,dsi,hri,water_perm,water_instab,recurrence_ratio,seasonal_water,esa_change_intensity,wb_x_impervious,eci_x_impervious,gsw_occ_x_impervious
latitude,1.000000,0.624468,-0.008186,-0.008266,0.034533,-0.008151,-0.008176,0.054291,-0.011862,0.029326,...,0.156446,0.137433,0.044747,0.088820,-0.123671,-0.031691,-0.074657,-0.022001,0.046817,0.058524
longitude,0.624468,1.000000,-0.041832,-0.038242,-0.010196,-0.043382,-0.041804,-0.002020,0.017278,-0.055236,...,0.039922,-0.027000,-0.183209,-0.110129,-0.110029,-0.212531,0.023355,0.033233,0.019617,0.063532
gaia_changed_ever_frac,-0.008186,-0.041832,1.000000,0.998205,0.661813,0.990930,0.999999,0.114104,-0.109352,-0.076120,...,0.004585,0.006698,-0.055838,-0.025777,-0.156020,-0.049365,-0.009109,-0.698230,0.204525,0.475113
gaia_impervious_frac_by_sample_year,-0.008266,-0.038242,0.998205,1.000000,0.662181,0.993541,0.998177,0.112587,-0.107954,-0.075351,...,0.005700,0.006101,-0.052796,-0.020204,-0.145212,-0.048761,-0.007191,-0.703732,0.209694,0.489289
gaia_recent_change_5y_frac,0.034533,-0.010196,0.661813,0.662181,1.000000,0.636831,0.662120,0.071262,-0.071484,-0.056023,...,0.015248,0.019632,-0.054133,-0.061526,-0.270178,-0.028727,-0.038268,-0.408895,0.133811,0.130653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
seasonal_water,-0.031691,-0.212531,-0.049365,-0.048761,-0.028727,-0.048493,-0.049370,-0.267551,0.275127,0.481222,...,0.017731,0.073244,0.773830,0.567187,0.085213,1.000000,-0.034087,0.040822,-0.012350,-0.032351
esa_change_intensity,-0.074657,0.023355,-0.009109,-0.007191,-0.038268,-0.000922,-0.009191,0.041532,-0.041708,-0.067812,...,-0.033376,0.011857,-0.072055,-0.093445,-0.036940,-0.034087,1.000000,0.004128,-0.005362,-0.009898
wb_x_impervious,-0.022001,0.033233,-0.698230,-0.703732,-0.408895,-0.698259,-0.698253,-0.049239,0.031215,-0.002993,...,-0.010672,-0.100129,0.024077,-0.040151,-0.023992,0.040822,0.004128,1.000000,-0.232445,-0.588679
eci_x_impervious,0.046817,0.019617,0.204525,0.209694,0.133811,0.209616,0.204477,0.001643,0.011123,0.029546,...,0.299245,-0.077081,-0.002497,0.031533,0.006041,-0.012350,-0.005362,-0.232445,1.000000,0.241725


Highly correlated pairs (|r| >= 0.9): 54


,feature_a,feature_b,abs_corr
1877,def,etgap,1.000000
2833,esa_change_count,esa_change_intensity,1.000000
4158,gaia_changed_ever_frac_1km,gaia_transition_year_mean_changed_pixels_1km,1.000000
164,gaia_changed_ever_frac,gaia_transition_year_mean_changed_pixels,0.999999
5493,dsi,eci,0.999941
3779,gsw_seasonality_mean_1km,water_instab,0.999615
4155,gaia_changed_ever_frac_1km,gaia_impervious_frac_by_sample_year_1km,0.999557
4237,gaia_impervious_frac_by_sample_year_1km,gaia_transition_year_mean_changed_pixels_1km,0.999551
161,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,0.998205
243,gaia_impervious_frac_by_sample_year,gaia_transition_year_mean_changed_pixels,0.998177
